## A Collection of Strategies: Tests in Turkish Stock Exchange(BIST)

Installing libraries and writing basic data pipeline, return, log return codes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import os 

tickers = pd.read_csv("tickers.csv", skiprows=[1])["Ticker"].tolist()

def get_clean_data(ticker, start_date="2025-01-01", end_date="2025-12-31"):
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)

    # yfinance can return MultiIndex columns; flatten to plain OHLCV names.
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data = data.ffill().bfill()
    data["Returns"] = data["Close"].pct_change()
    data["Log_Returns"] = np.log(data["Close"] / data["Close"].shift(1))
    data["Volatility"] = data["Log_Returns"].rolling(window=20).std() * np.sqrt(252)
    data["MA_20"] = data["Close"].rolling(window=20).mean()
    data["MA_50"] = data["Close"].rolling(window=50).mean()
    data["MA_200"] = data["Close"].rolling(window=200).mean()
    data["Intraday_Volatility"] = (data["High"] - data["Low"]) / data["Open"]
    data["ATR"] = data["High"] - data["Low"]
    data.dropna(inplace=True)
    return data

df = get_clean_data(tickers[0])  # Get data for the first ticker
print(df.head())    

Price       Close   High    Low   Open     Volume   Returns  Log_Returns  \
Date                                                                       
2025-10-20  14.74  14.80  13.95  14.52  103480909  0.032213     0.031705   
2025-10-21  14.28  14.79  14.13  14.78   91838287 -0.031208    -0.031705   
2025-10-22  14.35  14.64  14.15  14.28   68730842  0.004902     0.004890   
2025-10-23  13.87  14.46  13.87  14.35   85950420 -0.033450    -0.034022   
2025-10-24  14.45  14.58  13.88  13.94  121358273  0.041817     0.040966   

Price       Volatility    MA_20    MA_50     MA_200  Intraday_Volatility  
Date                                                                      
2025-10-20    0.358902  14.0140  15.3388  15.816093             0.058540  
2025-10-21    0.358701  13.9720  15.2852  15.801816             0.044655  
2025-10-22    0.353743  13.9485  15.2290  15.786550             0.034314  
2025-10-23    0.364940  13.9175  15.1744  15.768735             0.041115  
2025-10-24    0.3

## Defining Market Regime

In [ ]:
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

def market_regime_analysis(ticker, start_date="2025-01-01", end_date="2025-12-31"):
    data = get_clean_data(ticker, start_date=start_date, end_date=end_date)

    # 3. Data Normalization
    # HMMs are sensitive to the scale of input features
    features = ["Log_Returns", "Volatility", "Intraday_Volatility"]
    missing = [col for col in features if col not in data.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}. Available columns: {list(data.columns)}")

    X = data[features].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 4. Model Training
    # 3 states typically represent: Bearish/Panic, Neutral/Choppy, and Bullish/Stable
    model = hmm.GaussianHMM(n_components=3, covariance_type="full", n_iter=1000, random_state=42)
    model.fit(X_scaled)

    # 5. Regime Prediction
    data["Regime"] = model.predict(X_scaled)

    regime_map = {
    0: "Bullish Trend (High Intraday)",
    1: "Quiet / Sideways",
    2: "High Uncertainty / Stress"
    }

    # Apply mapping to your dataframe
    data['Market_Type'] = data['Regime'].map(regime_map)
    

    # Summary of State Characteristics
    print("--- Regime Characteristics ---")
    return data.groupby("Regime")[["Log_Returns", "Volatility", "Intraday_Volatility"]].mean(), data["Market_Type"].tail(1) 

ticker = "XU100.IS"
regime_summary, market_type = market_regime_analysis(ticker)
print(regime_summary)
print(market_type)

--- Regime Characteristics ---
Price   Log_Returns  Volatility  Intraday_Volatility
Regime                                              
0          0.021305    0.181519             0.027672
1         -0.000960    0.142899             0.010435
2          0.000455    0.215690             0.015579
Date
2025-12-30    Quiet / Sideways
Name: Market_Type, dtype: str


# Karma Strategies

In [ ]:
def calculate_vwap(data, period=20):
    """
    Calculate the Volume Weighted Average Price (VWAP) for a given period.
    """
    # Calculate the typical price with min_periods=1 to avoid NaN values at the start
    typical_price = (data['Close'] * data['Volume']).rolling(window=period, min_periods=1).sum() / data['Volume'].rolling(window=period, min_periods=1).sum()
    
    data['VWAP'] = typical_price
    return data['VWAP']


def vwap_strategy(data, vwap_period=20, market_regime=None):
    """
    Implement a simple trading strategy based on VWAP.
    Only trade in bullish regimes.
    """
    data['VWAP'] = calculate_vwap(data, vwap_period)  # Renamed for clarity
    data['Signal'] = 0  # Default: no signal
    
    if market_regime == "Quiet / Sideways":
        data['Signal'] = pd.cut(data['Close'] > data['VWAP'], 
                                bins=[False, True], 
                                labels=[-1, 1]).astype(int)
        # Or simpler:
        data['Signal'] = (data['Close'] > data['VWAP']).astype(int) * 2 - 1
    
    return data

In [45]:
#Universal Backtesting Framework
def backtest_strategy(data, initial_capital=100000):
    """
    Backtest a trading strategy based on generated signals.
    """
    # Check if data has enough rows
    if len(data) == 0:
        return None, None, None
    
    
    data['Position'] = data['Signal'].shift(1)  # Take position based on previous day's signal
    data['Position'] = data['Position'].fillna(0)  # No position on the first day

    # Calculate daily returns from the strategy
    data['Strategy_Returns'] = data['Position'] * data['Log_Returns']
    
    # Calculate cumulative returns
    data['Cumulative_Strategy_Returns'] = (1 + data['Strategy_Returns']).cumprod() - 1
    
    # Calculate total return and annualized return
    total_return = data['Cumulative_Strategy_Returns'].iloc[-1]
    num_years = len(data) / 252  # Assuming 252 trading days in a year
    annualized_return = (1 + total_return) ** (1 / num_years) - 1
    
    return total_return, annualized_return, data[['Close', 'VWAP', 'Signal', 'Cumulative_Strategy_Returns']]

#Universal Backtesting of Multiple Tickers and Average Return Calculation

def backtest_multiple_tickers(tickers, start_date="2025-01-01", end_date="2025-12-31", market_regime=None):
    results = []
    
    for ticker in tickers:
        try:
            print(f"Backtesting {ticker}...")
            data = get_clean_data(ticker, start_date, end_date)
            if len(data) == 0:
                print(f"  Skipping {ticker}: No data available after cleaning")
                continue
            data = vwap_strategy(data, vwap_period=20, market_regime=None)
            total_return, annualized_return, _ = backtest_strategy(data)
            if total_return is None:
                print(f"  Skipping {ticker}: Insufficient data for backtest")
                continue
            results.append({
                'Ticker': ticker,
                'Total Return': total_return,
                'Annualized Return': annualized_return
            })
        except Exception as e:
            print(f"  Error backtesting {ticker}: {str(e)}")
            continue
    
    return pd.DataFrame(results)

summary, regime = market_regime_analysis("XU100.IS")
backtest_results = backtest_multiple_tickers(tickers[:30], market_regime=regime)
print(backtest_results)

--- Regime Characteristics ---
Backtesting AEFES.IS...
Backtesting AGHOL.IS...
Backtesting AKBNK.IS...
Backtesting AKSA.IS...
Backtesting AKSEN.IS...
Backtesting ALARK.IS...
Backtesting ALTNY.IS...
Backtesting ANSGR.IS...
Backtesting ARCLK.IS...
Backtesting ASELS.IS...
Backtesting ASTOR.IS...
Backtesting BALSU.IS...
Backtesting BIMAS.IS...
Backtesting BRSAN.IS...
Backtesting BRYAT.IS...
Backtesting BSOKE.IS...
Backtesting BTCIM.IS...
Backtesting CANTE.IS...
Backtesting CCOLA.IS...
Backtesting CIMSA.IS...
Backtesting CVKMD.IS...
Backtesting CWENE.IS...
Backtesting DAPGM.IS...
Backtesting DOAS.IS...
Backtesting DOHOL.IS...
Backtesting DSTKF.IS...
Backtesting ECILC.IS...
Backtesting EFOR.IS...
Backtesting EKGYO.IS...
Backtesting ENERY.IS...
      Ticker  Total Return  Annualized Return
0   AEFES.IS           0.0                0.0
1   AGHOL.IS           0.0                0.0
2   AKBNK.IS           0.0                0.0
3    AKSA.IS           0.0                0.0
4   AKSEN.IS          

In [46]:
print(backtest_results.describe())

       Total Return  Annualized Return
count          30.0               30.0
mean            0.0                0.0
std             0.0                0.0
min             0.0                0.0
25%             0.0                0.0
50%             0.0                0.0
75%             0.0                0.0
max             0.0                0.0


# A Simplier Strategy & Backtesting & Regime

In [ ]:
def is_bullish_regime(start_date="2025-01-01", end_date="2025-12-31"):
    # checking bist30 and bist100 close price is above their 200-day moving average
    bist30 = get_clean_data("XU100.IS", start_date, end_date)
    bist100 = get_clean_data("XU030.IS", start_date, end_date)
    if bist30["Close"].iloc[-1] > bist30["MA_200"].iloc[-1] and bist100["Close"].iloc[-1] > bist100["MA_200"].iloc[-1]:
        return True
    return False

def mixed_strategy(tickers, start_date="2025-01-01", end_date="2025-12-31"):
    if is_bullish_regime(start_date, end_date):
        print("Bullish regime detected. Running strategy on all tickers.")
        for ticker in tickers:
            #is avg volume of that ticker > 10million and close price > sma200 and sma50> sma200 then run strategy
            data = get_clean_data(ticker, start_date, end_date)
            if data["Volume"].mean() > 10000000 and data["Close"].iloc[-1] > data["MA_200"].iloc[-1] and data["MA_50"].iloc[-1] > data["MA_200"].iloc[-1]:
                print(f"Running strategy for {ticker} with average volume {data['Volume'].mean()}")


    else:
        print("Not a bullish regime. No trading strategy applied.")
        return pd.DataFrame()  # Return empty DataFrame if not bullish
    